# 3 — Restructuring operations

Ops that change the *shape of the index space* while still only rewriting the
layout — now with charts showing what blocking means physically.

In [1]:
import numpy as np
from nbhelp import formula, show  # also puts tensorlib on sys.path
from tensorlib import Tensor, q, u

## split — blocking a dimension, physically

`x → (xb, xi)` via `x = const + Σ w_j·i_j`; strides scale by the weights.
With a chart, the outermost part inherits a **position** chart whose step is
the block pitch, and inner parts get **displacement** charts — physically:
*block position + offset within the block*.

A 16-sample line scan at 0.25 um pitch, tiled into 1 um blocks:

In [2]:
arr = np.arange(16, dtype=np.int64)
w = Tensor.from_numpy(arr, ("x",)).with_charts(x=("0 um", "0.25 um"))
show(w, "16 samples at 0.25 um")

-- 16 samples at 0.25 um
Tensor[int64] on Buffer(128B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  x            8      0     16     16  pos[x: 0 um step 0.25 um]
  numel=16  footprint=(0, 128)  injectivity=injective


In [3]:
b = w.split("x", xb=4, xi=4)
show(b, "blocked: xb = block position (1 um pitch), xi = offset in block")

-- blocked: xb = block position (1 um pitch), xi = offset in block
Tensor[int64] on Buffer(128B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  xb          32      0      4      4  pos[x: 0 um step 1 um]
  xi           8      0      4      4  disp[x: 0 um step 0.25 um]
  numel=16  footprint=(0, 128)  injectivity=injective


In [4]:
# sample 11 = block at 2 um, plus 0.75 um into the block
print(b.item(xb=q("2 um"), xi=q("0.75 um")), "==", arr[11])

11 == 11


## compiler mode: lattice splits with natural ranges

Everything works identically with plain ints — including caller-chosen index
ranges (blocks named 1..3 here). The constant folds into the offset.

In [5]:
t46 = Tensor.from_numpy(np.arange(24, dtype=np.int64).reshape(4, 6), ("i", "j"))
n = t46.split("j", jb=(1, 4), jw=2)
show(n, "j -> jb in [1,4), jw in [0,2)")
formula(n)

-- j -> jb in [1,4), jw in [0,2)
Tensor[int64] on Buffer(192B @ cpu)
  offset : -16 bytes
  dim     stride  start   stop   size  chart
  i           48      0      4      4  
  jb          16      1      4      3  
  jw           8      0      2      2  
  numel=24  footprint=(0, 192)  injectivity=injective
loc = -16 + 48*i + 16*jb + 8*jw   (bytes)


## merge — the inverse, restoring the chart exactly

Merging requires the byte strides *and* the chart steps to nest affinely;
the original chart is reconstructed, not remembered.

In [6]:
m = b.merge(("xb", "xi"), "x")
print("chart restored:", m.layout.dim("x").chart == w.layout.dim("x").chart)
print("values intact: ", bool(np.array_equal(m.to_numpy(), arr)))

chart restored: True
values intact:  True


## decimate — every k-th element, labels intact

Decimation renumbers the lattice (it must — domains are dense boxes), but
the chart keeps the physical truth: origin gains `phase·step`, step scales by
the factor. Physical indexing is unaffected.

In [7]:
dec = w.decimate("x", 4, phase=1)
show(dec, 'w.decimate("x", 4, phase=1)')
print(dec.item(x=q("1.25 um")), "==", arr[5])

-- w.decimate("x", 4, phase=1)
Tensor[int64] on Buffer(128B @ cpu)
  offset : 8 bytes
  dim     stride  start   stop   size  chart
  x           32      0      4      4  pos[x: 0.25 um step 1 um]
  numel=4  footprint=(8, 112)  injectivity=injective
5 == 5


Decimate is exactly split + select — and thanks to axis identity, the
select *compensates*: selecting the phase folds its 0.25 um label into the
block dim's origin. Selecting the block instead promotes the in-block
offset to absolute positions.

In [8]:
via_split = b.select(xi=q("0.25 um"))  # select the in-block offset: decimation
print("same chart as decimate:", via_split.layout.dim("xb").chart == dec.layout.dim("x").chart)
blk = b.select(xb=q("2 um"))  # select a block: absolute positions
print(blk.layout.dim("xi").chart)
print(blk.item(xi=q("2.75 um")), "==", arr[11])

same chart as decimate: True
pos[x: 2 um step 0.25 um]
11 == 11


## alignment — diagnosis, never surgery

Elementwise computation will require operands on a common lattice — but
there is deliberately no op that aligns them for you (the same reasoning as
no-permute: bundling semantic choices teaches users not to make them).
`alignment` reports exactly what is wrong and the primitive that fixes each
item; *you* spend the primitives, consciously. Empty report = aligned.

In [9]:
from tensorlib import aligned, alignment

other = Tensor.from_numpy(np.arange(16, dtype=np.int64) * 100, ("x",))
other = other.with_charts(x=("0.5 um", "0.25 um"))  # same pitch, offset grid
for m in alignment(w, other):
    print(m)

operand 1, dim 'x': frames offset by 2 steps  ->  shift(x=2)


In [10]:
other2 = other.shift(x=2)  # apply the recipe, then re-run the diagnosis
for m in alignment(w, other2):
    print(m)

operand 0, dim 'x': domain [0, 16) exceeds the common [2, 16)  ->  slice(x=(2, 16))
operand 1, dim 'x': domain [2, 18) exceeds the common [2, 16)  ->  slice(x=(2, 16))


In [11]:
va, vb = w.slice(x=(2, 16)), other2.slice(x=(2, 16))
print("aligned:", aligned(va, vb))
print(va.item(x=q("1 um")), vb.item(x=q("1 um")))

aligned: True
4 200


## diagonal — n-ary, and it never guesses your labels

`stride_z = sum of the strides`; the domain is the intersection; guard
coefficients sum — and any number of dims may be consumed at once. Charts
(D16): parts sharing one axis get the forced label-sum chart; anything else
is **uncharted** until you say what z measures — a Chart, or a combinator
receiving the consumed dims.

In [12]:
img = Tensor.from_numpy(np.arange(16, dtype=np.int64).reshape(4, 4), ("i", "j"))
img = img.with_charts(i=("0 um", "6.5 um"), j=("0 um", "6.5 um"))
d = img.diagonal(("i", "j"), "z")
show(d, "different axes: z is uncharted by default")
d2 = img.diagonal(("i", "j"), "z", chart=img.layout.dim("i").chart)
print(d2.item(z=q("13 um")), "== pixel (2,2) =", img.item(i=2, j=2))

-- different axes: z is uncharted by default
Tensor[int64] on Buffer(128B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  z           40      0      4      4  
  numel=4  footprint=(0, 128)  injectivity=injective
10 == pixel (2,2) = 10


The physically rich case is a **rate diagonal** — a wave characteristic
x = c·t. `characteristic` is a chart combinator: it validates the CFL-style
commensurability condition exactly (step_x == rate · step_t) and labels z
along the axis you choose. All the physics is in a named, testable object;
`diagonal` itself stays dumb.

In [13]:
from tensorlib import characteristic

wave = Tensor.from_numpy(np.arange(16, dtype=np.int64).reshape(4, 4), ("x", "t"))
wave = wave.with_charts(x=("0 um", "0.5 um"), t=("0 ms", "0.25 ms"))
xi = wave.diagonal(("x", "t"), "xi", chart=characteristic(q("2 um/ms"), "x"))
show(xi, "characteristic at c = 2 um/ms")
try:
    wave.diagonal(("x", "t"), "xi", chart=characteristic(q("3 um/ms"), "x"))
except ValueError as e:
    print("wrong rate:", e)

-- characteristic at c = 2 um/ms
Tensor[int64] on Buffer(128B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  xi          40      0      4      4  pos[x: 0 um step 0.5 um]
  numel=4  footprint=(0, 128)  injectivity=injective
wrong rate: lattice is not characteristic for rate 3 um/ms: step(x) = 0.5 um but rate * step(t) = 0.75 um/ms*ms


## window — overlapping views with physically labeled taps

The kernel dim shares `x`'s stride and gets a **displacement** chart with
`x`'s step: taps are literally labeled −1 ms, 0 ms, +1 ms. `x`'s domain
shrinks so every tap stays in bounds (window has no fill — that is the
stencil, notebook 4).

In [14]:
ts = Tensor.from_numpy(np.arange(6, dtype=np.int64) * 10, ("t",))
ts = ts.with_charts(t=("0 ms", "1 ms"))
wnd = ts.window("t", "k", (q("-1 ms"), q("2 ms")))  # half-open k range
show(wnd, "centered 3-tap window")
print(wnd.item(t=q("2 ms"), k=q("-1 ms")), "==", ts.item(t=q("1 ms")))

-- centered 3-tap window
Tensor[int64] on Buffer(48B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  t            8      1      5      4  pos[t: 0 ms step 1 ms]
  k            8     -1      2      3  disp[t: 0 ms step 1 ms]
  numel=12  footprint=(0, 48)  injectivity=unknown
10 == 10


## field selection — and units for VALUES

Coordinates are not the only thing with units: `value_units` labels the value
space, per-field for structured dtypes. It is metadata for the future compute
layer (dimensional checking) — `item()` still returns raw machine numbers,
because stored values are inexact floats, and the exact Quantity type is
reserved for coordinates.

In [15]:
u.define("V", dim="voltage")
dt = np.dtype([("v", "<f4"), ("flags", "u1")], align=True)
rec = np.zeros(3, dt)
rec["v"] = [1.5, 2.5, 3.5]
rec["flags"] = [0, 1, 0]
probe = Tensor.from_numpy(rec, ("t",)).with_charts(t=("0 ms", "0.5 ms")).with_value_units({"v": u.V})
show(probe, "probe record: charted time, per-field value units")

-- probe record: charted time, per-field value units
Tensor[{'names': ['v', 'flags'], 'formats': ['<f4', 'u1'], 'offsets': [0, 4], 'itemsize': 8, 'aligned': True}] on Buffer(24B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  t            8      0      3      3  pos[t: 0 ms step 0.5 ms]
  numel=3  footprint=(0, 24)  injectivity=injective
  values : {'v': V}


In [16]:
fv = probe.field("v")
show(fv, 'field("v"): dtype narrows to float32, value unit narrows to V')
fv.item(t=q("1 ms"))  # a raw float32, not a Quantity

-- field("v"): dtype narrows to float32, value unit narrows to V
Tensor[float32] on Buffer(24B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  t            8      0      3      3  pos[t: 0 ms step 0.5 ms]
  numel=3  footprint=(0, 20)  injectivity=injective
  values : V


np.float32(3.5)

## categorical labels — the nominal rung

Often a dim has concrete named points of reference before it has any
numerical scale — R/G/B channels, site names. `with_labels` attaches that
nominal labeling: a bijection between lattice and names, with NO arithmetic
(no step, no displacements). Ops that need arithmetic (split, window,
stencil, pad, diagonal) refuse labeled dims; slice / shift / flip /
decimate / select thread the names glued to the data.

In [17]:
rgb = Tensor.from_numpy(np.arange(12, dtype=np.int64).reshape(3, 4), ("c", "x"))
rgb = rgb.with_labels(c=("R", "G", "B")).with_charts(x=("0 um", "6.5 um"))
show(rgb, "an RGB scanline")
print(rgb.item(c="G", x=q("13 um")), "==", rgb.item(c=1, x=2))

-- an RGB scanline
Tensor[int64] on Buffer(96B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  c           32      0      3      3  #[R, G, B]
  x            8      0      4      4  pos[x: 0 um step 6.5 um]
  numel=12  footprint=(0, 96)  injectivity=injective
6 == 6


In [18]:
print(rgb.flip("c").layout.dim("c").labels, " (glued through the flip)")
try:
    rgb.window("c", "k", 2)
except TypeError as e:
    print("refused:", e)

('B', 'G', 'R')  (glued through the flip)
refused: dim c is categorical (labels ('R', 'G', 'B')); window has no meaning on nominal data — attach a chart first if a numeric scale exists


And the upgrade path: once a numerical scale exists, attaching a chart
REPLACES the labels — nominal -> interval, while the lattice and the data
never move. (Non-uniform coordinates — city longitudes, RGB center
wavelengths — are a third rung the affine family doesn't cover yet; see
CONCERNS #14.)

In [19]:
show(rgb.with_charts(c=("0 ms", "5 ms")), "the same dim, now on a time scale")

-- the same dim, now on a time scale
Tensor[int64] on Buffer(96B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  c           32      0      3      3  pos[c: 0 ms step 5 ms]
  x            8      0      4      4  pos[x: 0 um step 6.5 um]
  numel=12  footprint=(0, 96)  injectivity=injective
